# 🔴 Interview: QLoRA Linear Layer

Quantized base weight + low-rank adapter for efficient fine-tuning

In [ ]:
import torch
import torch.nn as nn


In [ ]:
# ✅ INTERVIEW

class QLoRALinear(nn.Module):
    """
    手写 QLoRA —— 面试版。

    面试考点:
    1. QLoRA = 量化基础权重 + LoRA 适配器
    2. int8 量化: scale = max(|W|)/127, q = round(W/scale).clamp(-127,127)
    3. LoRA: ΔW = A @ B, A 初始化随机, B 初始化为零
    4. 前向: dequant(base) + x@A@B*(alpha/rank)

    参数: in_features, out_features, rank, alpha=1.0
    """
    def __init__(self, in_features, out_features, rank, alpha=1.0):
        super().__init__()
        self.rank = rank
        self.alpha = alpha

        # 量化权重 (int8, 不需要梯度)
        self.quantized_weight = nn.Parameter(
            torch.zeros(out_features, in_features, dtype=torch.int8),
            requires_grad=False
        )
        # 每行缩放因子
        self.scale = nn.Parameter(torch.ones(out_features, 1))
        # LoRA A: (in, rank), 小随机初始化
        self.lora_A = nn.Parameter(torch.randn(in_features, rank) * 0.01)
        # LoRA B: (rank, out), 零初始化 → 训练初期 ΔW=0
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))

    def set_weight(self, W_fp32):
        """量化浮点权重: scale = max(|W|)/127, q = round(W/scale).clamp(-127,127)"""
        scale = W_fp32.abs().max(dim=1, keepdim=True).values.clamp(min=1e-8) / 127.0
        q = (W_fp32 / scale).round().clamp(-127, 127).to(torch.int8)
        self.quantized_weight.data.copy_(q)
        self.scale.data.copy_(scale)

    def forward(self, x):
        """
        x: (B, in) → (B, out)
        base = x @ dequant(W).T
        delta = x @ A @ B * (alpha / rank)
        return base + delta
        """
        # 反量化: int8 → float → 乘以 scale
        W_fp = self.quantized_weight.float() * self.scale  # (out, in)
        # 基础输出
        base = x @ W_fp.T  # (B, out)
        # LoRA 增量
        delta = x @ self.lora_A @ self.lora_B * (self.alpha / self.rank)
        return base + delta

In [ ]:
torch.manual_seed(0)
in_f, out_f, rank = 64, 32, 4
layer = QLoRALinear(in_f, out_f, rank, alpha=2.0)

# Set weight from a full-precision reference
W_ref = torch.randn(out_f, in_f)
layer.set_weight(W_ref)

x = torch.randn(8, in_f)
y_qlora = layer(x)
y_ref = x @ W_ref.T  # full-precision baseline (no LoRA delta)

print("Output shape:", y_qlora.shape)          # (8, 32)
print("Max abs error vs fp32:", (y_qlora - y_ref).abs().max().item())

In [ ]:
from torch_judge import check
check("qlora")

## 📝 核心思路总结

1. **QLoRA = 量化 + LoRA**：基础权重用 int8 存储（模拟 4-bit），LoRA 适配器保持 fp32，兼顾内存效率和微调质量。
2. **量化公式**：`scale = max(|W|)/127`，`q = round(W/scale).clamp(-127,127)`，每行独立缩放，保留相对精度。
3. **LoRA 的零初始化**：B 初始化为零，训练开始时 ΔW=0，模型行为与纯量化模型一致，渐进学习任务特定信息。
4. **alpha/rank 缩放**：`alpha/rank` 控制 LoRA 更新的幅度，rank 越大单个参数影响越小，防止低秩更新过大扰动基础模型。